<a href="https://colab.research.google.com/github/Michael321826/BotForge/blob/main/murf_python_sdk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🗣️ Murf Stream Speech – Interactive Colab Demo

Welcome! This notebook lets you **try Murf's low-latency streaming Text-to-Speech (TTS)** with our `FALCON` model.

You can:

1. Set up the Murf SDK
2. Make your **first streaming request**
3. Hear the audio directly **inside this notebook**
4. Use a **playground UI** to tweak voice, locale, rate, pitch, variation & style
5. (Optional) See how to call the **raw HTTP streaming endpoint**

> **You’ll need a Murf API key** from your Murf dashboard before you start - <a href="https://murf.ai/api/dashboard" target="_blank">**Get Murf API Key**</a>



## How to use this notebook

1. **Install dependencies**  
   Run the cell titled **`1. Install dependencies`**  
   - This installs the Murf SDK and Colab UI widgets.

2. **Configure your API key & region**  
   Run **`2. Configure your Murf API key & region`**  
   - Paste your Murf API key into `MURF_API_KEY`, **or**  
   - Set `MURF_API_KEY` empty and toggle `USE_ENV_VAR = True` and use the `MURF_API_KEY` environment variable.

3. **Enable audio helper functions**  
   Run **`3. Helper: convert streaming bytes to a WAV file & preview in notebook`**  
   This cell:
   - Collects streamed audio chunks
   - Writes them to a small WAV file (in this notebook’s workspace)
   - Lets you **play audio inline** in the notebook.

4. **Try a simple streaming call**  
   Run **`4. Your first streaming request`**  
   This:
   - Sends your text to Murf using `model="FALCON"`
   - Streams back audio
   - Plays that audio **right in the notebook UI**

5. **Use the playground UI (recommended)**  
   Run **`5. Explore voice, locale, pitch, rate & variation (playground)`**  
   Here you can:
   - Type any text  
   - Pick a **voice**  
   - Pick a **locale**  
   - Adjust **rate**, **pitch**, **variation**, **style**  
   - Click **“Stream & Play”** to hear the result

6. **(Optional) Raw HTTP streaming**  
   Use **`6. (Optional) Call the streaming HTTP endpoint directly with requests`** if you want to:
   - Inspect the exact HTTP request / response
   - Port the integration to languages other than Python.

---

### Streaming vs non-streaming

- This notebook is focused on **streaming speech** with `Stream Speech (Beta)` + `model="FALCON"`.
- If you want:
  - A **voice browser** (listen to lots of voices / locales quickly), or
  - Classic **non-streaming TTS** (one-shot MP3/WAV, no streaming),
  
  use the **linked notebooks** at the bottom under **“Next steps & related notebooks”**.


# 1. Install Dependencies

In [ ]:
pip install --quiet murf ipywidgets

Note: you may need to restart the kernel to use updated packages.


# 2. Configure your Murf API key & region

In [ ]:
import os
import ipywidgets as widgets
from murf import Murf, MurfRegion

# Paste your API key here OR set it in the environment before running the notebook.
MURF_API_KEY = "YOUR_API_KEY" # Get Your Murf API key from your Murf API dashboard here - https://murf.ai/api
USE_ENV_VAR = False

# If you prefer environment variables, set MURF_API_KEY above to empty string and toggle USE_ENV_VAR to True.
if USE_ENV_VAR and not MURF_API_KEY:
    MURF_API_KEY = os.getenv("MURF_API_KEY")

if not MURF_API_KEY:
    raise ValueError(
        "No API key set. Either set MURF_API_KEY above or as an environment variable."
    )

# Choose region for lower latency (GLOBAL auto-routes to nearest region)
REGION_NAME = "GLOBAL"

# Map string to MurfRegion enum
region_map = {
    "GLOBAL": MurfRegion.GLOBAL,
    "India": MurfRegion.IN,
    "US-EAST": MurfRegion.US_EAST,
    "US_WEST": MurfRegion.US_WEST,
    "Europe": MurfRegion.EU_CENTRAL,
    "Japan": MurfRegion.JP,
    "Australia": MurfRegion.AU,
    "Korea": MurfRegion.KR,
    "Middle East": MurfRegion.ME,
    "South America": MurfRegion.SA_EAST,
    "UK": MurfRegion.UK,
    "Canada": MurfRegion.CA,
}
region = region_map[REGION_NAME]

client = Murf(
    api_key=MURF_API_KEY,
    region=region,
)

print(f"Murf client initialized with region = {REGION_NAME}")

Murf client initialized with region = GLOBAL


# 3. Helper: Convert Streaming Bytes to a WAV File & Preview in Notebook

In [ ]:
import wave
from typing import Iterable
from IPython.display import Audio, display

SAMPLE_RATE = 24000
CHANNELS = 1
SAMPLE_WIDTH_BYTES = 2


def stream_to_wav(
    audio_stream: Iterable[bytes],
    sample_rate: int = SAMPLE_RATE,
    path: str = "./murf_stream_output.wav",
) -> str:
    with wave.open(path, "wb") as wf:
        wf.setnchannels(CHANNELS)
        wf.setsampwidth(SAMPLE_WIDTH_BYTES)
        wf.setframerate(sample_rate)
        for chunk in audio_stream:
            wf.writeframes(chunk)
    return path


def play_stream_in_notebook(audio_stream: Iterable[bytes], filename: str = "./murf_stream_output.wav"):
    wav_path = stream_to_wav(audio_stream, SAMPLE_RATE, filename)
    display(Audio(wav_path, autoplay=True))


# 4. Your First Streaming Request

In [ ]:
import time
default_text = "Hello, how are you doing today?"
voice_id = "Matthew"
model = "FALCON"
locale = "en-US"

print("Requesting streaming audio from Murf...")

start_time=time.time()
audio_chunks = []
first_byte_time = None

audio_stream = client.text_to_speech.stream(
    text=default_text,
    voice_id=voice_id,
    model=model,
    locale=locale,
    sample_rate=SAMPLE_RATE,
    format="PCM",
)

for chunk in audio_stream:
    if first_byte_time is None:
        first_byte_time = time.time()
        ttfb_ms = (first_byte_time - start_time) * 1000
        print(f"Time-to-First-Byte (TTFB): {ttfb_ms:.2f} ms\n")
    audio_chunks.append(chunk)

play_stream_in_notebook(audio_chunks, filename="./murf_playground_stream.wav")


Requesting streaming audio from Murf...
Time-to-First-Byte (TTFB): 150.78 ms



# 5. Explore Voice, Locale, Pitch, Rate & Variation (playground)

## Streaming Playground – voices, locales & controls

Use the UI below to **experiment with Falcon streaming TTS**:

1. Type your text in the big **Text** box.
2. Choose a **Voice ID** that matches the language / style you want.
3. Choose a **Locale** (e.g., `en-US`, `en-IN`, `fr-FR`).
4. Adjust:
   - `rate` → speed (negative = slower, positive = faster)
   - `pitch` → lower / higher tone
   - `variation` → how much the model can express differently (0–5)
   - `style` → e.g., `Conversation`, `Angry` (only if supported by that voice/model)
5. Click **"Stream & Play"** and wait for:
   - A status message in the output
   - The audio player to appear **below the button**

> If you don’t hear anything:
> - Make sure you ran the **setup cells above**  
> - Check that your **API key is valid**  
> - Try with default parameters


In [ ]:
import requests
from IPython.display import display

BASE_URL = "https://api.murf.ai"

resp = requests.get(
    f"{BASE_URL}/v1/speech/voices",
    params={"model": "falcon"},
    headers={"api-key": MURF_API_KEY,}
)
resp.raise_for_status()
voices_list = resp.json()

# -----------------------------
# 1. Build config from API data
# -----------------------------

def build_voice_config(voices):
    """
    Transform /v1/speech/voices?model=falcon response into:
      - voice_config: per-voice supported locales + styles
      - locale_index: per-locale list of voices that support it
    """
    voice_config = {}
    locale_index = {}

    for v in voices:
        voice_id = v.get("voiceId")
        if not voice_id:
            continue

        display_name = v.get("displayName", voice_id)  # e.g. "Alina (F)"
        supported_locales = v.get("supportedLocales", {}) or {}
        fallback_styles = v.get("availableStyles", []) or []

        voice_cfg_locales = {}

        for locale_code, loc_info in supported_locales.items():
            locale_label = loc_info.get("detail", locale_code)
            styles = loc_info.get("availableStyles", []) or fallback_styles or ["Conversation"]

            voice_cfg_locales[locale_code] = {
                "locale_label": locale_label,
                "styles": styles,
            }

            if locale_code not in locale_index:
                locale_index[locale_code] = {
                    "label": locale_label,
                    "voices": [],
                }
            locale_index[locale_code]["voices"].append({
                "voiceId": voice_id,
                "displayName": display_name,
            })

        if voice_cfg_locales:
            voice_config[voice_id] = {
                "display_name": display_name,
                "supported_locales": voice_cfg_locales,
            }

    return voice_config, locale_index


voice_config, locale_index = build_voice_config(voices_list)


# --------------------------------
# 2. Defaults
# --------------------------------

DEFAULT_LOCALE_CODE = "en-US"
DEFAULT_STYLE = "Conversation"
DEFAULT_VOICE_SUBSTRING = "matthew"

def pick_default_locale():
    if DEFAULT_LOCALE_CODE in locale_index:
        return DEFAULT_LOCALE_CODE
    return sorted(locale_index.keys())[0]

def pick_default_voice_for_locale(locale_code):
    voices_for_locale = locale_index.get(locale_code, {}).get("voices", [])
    if not voices_for_locale:
        return None

    for v in voices_for_locale:
        vid = v["voiceId"].lower()
        dname = v["displayName"].lower()
        if DEFAULT_VOICE_SUBSTRING in vid or DEFAULT_VOICE_SUBSTRING in dname:
            return v["voiceId"]

    return voices_for_locale[0]["voiceId"]

def pick_default_style_for_voice_locale(voice_id, locale_code):
    voice = voice_config.get(voice_id, {})
    loc_cfg = voice.get("supported_locales", {}).get(locale_code, {})
    styles = loc_cfg.get("styles", []) or [DEFAULT_STYLE]
    for s in styles:
        if s.lower() == DEFAULT_STYLE.lower():
            return s
    return styles[0]


initial_locale_code = pick_default_locale()
initial_voice_id = pick_default_voice_for_locale(initial_locale_code)
initial_style = pick_default_style_for_voice_locale(initial_voice_id, initial_locale_code)


# --------------------------------
# 3. Widgets (Locale -> Voices -> Styles)
# --------------------------------

text_widget = widgets.Textarea(
    value="Welcome to the Murf streaming API playground!",
    description="Text:",
    layout=widgets.Layout(width="100%", height="100px"),
)

def build_locale_options():
    items = []
    for code, meta in locale_index.items():
        label = meta.get("label", code)
        items.append((label, code))
    items.sort(key=lambda t: t[0])
    return items

locale_widget = widgets.Dropdown(
    options=build_locale_options(),
    value=initial_locale_code,
    description="Locale:",
)

def build_voice_options_for_locale(locale_code):
    voices_for_locale = locale_index.get(locale_code, {}).get("voices", [])
    options = []
    for v in voices_for_locale:
        label = v["displayName"]
        options.append((label, v["voiceId"]))
    options.sort(key=lambda t: t[0])
    return options

voice_widget = widgets.Dropdown(
    options=build_voice_options_for_locale(initial_locale_code),
    value=initial_voice_id,
    description="Voice:",
)

def build_style_options(voice_id, locale_code):
    voice = voice_config.get(voice_id, {})
    loc_cfg = voice.get("supported_locales", {}).get(locale_code, {})
    styles = loc_cfg.get("styles", []) or [DEFAULT_STYLE]
    seen = set()
    uniq_styles = []
    for s in styles:
        if s not in seen:
            seen.add(s)
            uniq_styles.append(s)
    return uniq_styles

style_widget = widgets.Dropdown(
    options=build_style_options(initial_voice_id, initial_locale_code),
    value=initial_style,
    description="Style:",
)

rate_widget = widgets.IntSlider(
    value=0,
    min=-50,
    max=50,
    step=1,
    description="Rate:",
)

pitch_widget = widgets.IntSlider(
    value=0,
    min=-50,
    max=50,
    step=1,
    description="Pitch:",
)

variation_widget = widgets.IntSlider(
    value=1,
    min=0,
    max=5,
    step=1,
    description="Variation:",
)

button = widgets.Button(
    description="Stream & Play",
    button_style="primary",
)

output = widgets.Output()


# --------------------------------
# 4. Reactive behavior
# --------------------------------

def on_locale_change(change):
    if change["name"] != "value":
        return
    new_locale = change["new"]

    voice_options = build_voice_options_for_locale(new_locale)
    if not voice_options:
        voice_widget.options = []
        style_widget.options = []
        return

    voice_widget.options = voice_options
    current_voice = voice_widget.value
    valid_ids = [val for _, val in voice_options]
    if current_voice not in valid_ids:
        current_voice = valid_ids[0]
    voice_widget.value = current_voice

    style_options = build_style_options(current_voice, new_locale)
    style_widget.options = style_options

    default_style = pick_default_style_for_voice_locale(current_voice, new_locale)
    if default_style in style_options:
        style_widget.value = default_style
    else:
        style_widget.value = style_options[0]


def on_voice_change(change):
    if change["name"] != "value":
        return
    new_voice = change["new"]
    current_locale = locale_widget.value

    style_options = build_style_options(new_voice, current_locale)
    style_widget.options = style_options

    default_style = pick_default_style_for_voice_locale(new_voice, current_locale)
    if default_style in style_options:
        style_widget.value = default_style
    else:
        style_widget.value = style_options[0]


locale_widget.observe(on_locale_change, names="value")
voice_widget.observe(on_voice_change, names="value")


# --------------------------------
# 5. Streaming handler (uses codes, not labels)
# --------------------------------

def on_click(_):
    output.clear_output()
    with output:
        print("Making streaming request...")
        try:
            locale_code = locale_widget.value
            voice_id = voice_widget.value
            style = style_widget.value

            print(f"Voice: {voice_id} | Locale: {locale_code} | Style: {style}")

            audio_stream = client.text_to_speech.stream(
                text=text_widget.value,
                voice_id=voice_id,
                model="FALCON",
                locale=locale_code,
                sample_rate=SAMPLE_RATE,
                format="PCM",
                rate=rate_widget.value,
                pitch=pitch_widget.value,
                variation=variation_widget.value,
                style=style or None,
            )
            play_stream_in_notebook(audio_stream, filename="./murf_playground_stream.wav")
        except Exception as e:
            print("Error during streaming:", e)


button.on_click(on_click)

controls = widgets.VBox(
    [
        text_widget,
        widgets.HBox([locale_widget, voice_widget]),
        widgets.HBox([rate_widget, pitch_widget]),
        widgets.HBox([variation_widget, style_widget]),
        button,
    ]
)

display(controls, output)


Output()

# 6. (Optional) Call the streaming HTTP endpoint directly with requests

In [ ]:
import json
import requests

# You can pin to a specific region's base URL if desired:
#   Global (auto-routes): https://global.api.murf.ai/v1/speech/stream
#   US-East:              https://us-east.api.murf.ai/v1/speech/stream
#   India:                https://in.api.murf.ai/v1/speech/stream
#   See Ful list here - https://murf.ai/api/docs/api-reference/text-to-speech/stream

BASE_URL = "https://global.api.murf.ai/v1/speech/stream"

payload = {
    "text": "Hello, how are you doing today?",
    "voiceId": "Matthew",
    "model": "FALCON",
    "locale": "en-US",
    "sampleRate": SAMPLE_RATE,
    "format": "PCM",
}

headers = {
    "Content-Type": "application/json",
    "api-key": MURF_API_KEY,
}

start_time = time.time()

response = requests.post(
    BASE_URL,
    headers=headers,
    data=json.dumps(payload),
    stream=True,
)

if response.status_code != 200:
    print(f"❌ Error {response.status_code}: {response.text[:1000]}")
else:
    first = [True]

    def http_chunk_generator():
        for chunk in response.iter_content(chunk_size=None):
            if chunk:
                if first[0]:
                    ttfb = time.time() - start_time
                    print(f"TTFB: {ttfb:.3f} seconds")
                    first[0] = False
                yield chunk

    play_stream_in_notebook(http_chunk_generator(), filename="./murf_http_stream.wav")
    print("Done.")


TTFB: 0.254 seconds


Done.


---

## Related notebooks

This notebook is focused on **low-latency, streaming speech** with `FALCON`.

For other workflows, open these notebooks in a new tab:

- **Non-streaming Speech API**  
  _Colab link:_ [Open non-streaming speech API notebook](https://colab.research.google.com/gist/devgeetech-murf/bbe2c7eb01433f4a151f0fd2be23b1c8/murf-python-sdk.ipynb)

---